# Exercício 3: Gold de Engajamento por Turma

Notebook de exploração da tabela Delta gerada pelo pipeline `src/lakehouse_gold_engajamento_turma.py`.

Tabela: `data/gold/gold_engajamento_turma` (uma linha por `escola_id`/`turma_id`/`disciplina`/`mes_referencia`, com escrita idempotente: overwrite na primeira execução, `MERGE` nas seguintes).

In [1]:
import sys

# permite importar os módulos de src/ quando o notebook roda a partir de notebooks/
sys.path.insert(0, "../src")

from lakehouse_bronze_matriculas import create_spark_session

spark = create_spark_session(app_name="notebook_exercicio_3")
gold_path = "../data/gold/gold_engajamento_turma"

:: loading settings :: url = jar:file:/Users/jpdagostin/Desktop/personal/lakehouse_education/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/jpdagostin/.ivy2.5.2/cache
The jars for the packages stored in: /Users/jpdagostin/.ivy2.5.2/jars
io.delta#delta-spark_4.1_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-00e6aab8-7942-4325-b4ad-9aa7fbc02a8d;1.0
	confs: [default]
	found io.delta#delta-spark_4.1_2.13;4.3.1 in central
	found io.delta#delta-storage;4.3.1 in central
	found io.unitycatalog#unitycatalog-client;0.5.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.5.0 in central


	found org.apache.logging.log4j#log4j-core;2.25.3 in central
	found io.delta#delta-kernel-api;4.3.1 in central
	found org.roaringbitmap#RoaringBitmap;0.9.25 in central
	found com.fasterxml.jackson.core#jackson-databind;2.13.5 in central
	found com.fasterxml.jackson.core#jackson-annotations;2.13.5 in central
	found com.fasterxml.jackson.core#jackson-core;2.13.5 in central
	found com.fasterxml.jackson.datatype#jackson-datatype-jdk8;2.13.5 in central
	found org.roaringbitmap#shims;0.9.25 in central
	found io.delta#delta-kernel-defaults;4.3.1 in central
	found org.apache.hadoop#hadoop-client-runtime;3.4.2 in central
	found org.apache.parquet#parquet-hadoop;1.16.0 in central
	found org.apache.parquet#parquet-column;1.16.0 in central
	found org.apache.parquet#parquet-common;1.16.0 in central
	found org.apache.parquet#parquet-format-structures;1.16.0 in central
	found javax.annotation#javax.annotation-api;1.3.2 in central
	found org.apache.parquet#parquet-encoding;1.16.0 in central
	found org

26/08/12 22:52:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


## Schema da tabela

In [2]:
df_gold = spark.read.format("delta").load(gold_path)
df_gold.printSchema()

root
 |-- escola_id: string (nullable = true)
 |-- turma_id: string (nullable = true)
 |-- disciplina: string (nullable = true)
 |-- mes_referencia: date (nullable = true)
 |-- total_submissoes: long (nullable = true)
 |-- total_acertos: long (nullable = true)
 |-- taxa_acerto_pct: double (nullable = true)
 |-- tempo_medio_segundos: double (nullable = true)
 |-- alunos_distintos: long (nullable = true)
 |-- dt_processamento_gold: timestamp (nullable = true)



## Métricas agregadas por escola/turma/disciplina/mês

In [3]:
df_gold.orderBy("escola_id", "turma_id", "disciplina", "mes_referencia").show(truncate=False)

26/08/12 22:52:37 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+---------+--------+----------+--------------+----------------+-------------+-----------------+--------------------+----------------+--------------------------+
|escola_id|turma_id|disciplina|mes_referencia|total_submissoes|total_acertos|taxa_acerto_pct  |tempo_medio_segundos|alunos_distintos|dt_processamento_gold     |
+---------+--------+----------+--------------+----------------+-------------+-----------------+--------------------+----------------+--------------------------+
|esc01    |7A      |matematica|2026-02-01    |4               |3            |75.0             |151.25              |2               |2026-08-10 08:05:15.877445|
|esc01    |7A      |portugues |2026-02-01    |1               |1            |100.0            |110.0               |1               |2026-08-10 08:05:15.877445|
|esc01    |7B      |matematica|2026-02-01    |2               |1            |50.0             |275.0               |1               |2026-08-10 08:05:15.877445|
|esc02    |8A      |portugues |202

## Pergunta em linguagem natural: "qual turma teve a maior taxa de acerto?"

Exemplo do tipo de pergunta que essa tabela Gold já responde diretamente, sem precisar de outra fonte — é o caso de uso citado no enunciado do Exercício 3 (consumo via agente LLM/MCP).

In [4]:
from pyspark.sql import functions as F

df_gold.orderBy(F.desc("taxa_acerto_pct")).select(
    "escola_id", "turma_id", "disciplina", "mes_referencia", "taxa_acerto_pct", "total_submissoes"
).show(5, truncate=False)

+---------+--------+----------+--------------+-----------------+----------------+
|escola_id|turma_id|disciplina|mes_referencia|taxa_acerto_pct  |total_submissoes|
+---------+--------+----------+--------------+-----------------+----------------+
|esc01    |7A      |portugues |2026-02-01    |100.0            |1               |
|esc01    |7A      |matematica|2026-02-01    |75.0             |4               |
|esc02    |8A      |portugues |2026-02-01    |66.66666666666666|3               |
|esc01    |7B      |matematica|2026-02-01    |50.0             |2               |
+---------+--------+----------+--------------+-----------------+----------------+



## Histórico de versões (time travel via Delta log)

Cada execução do pipeline gera uma nova versão na tabela (overwrite na primeira execução, merge/upsert nas seguintes).

In [5]:
from delta.tables import DeltaTable

DeltaTable.forPath(spark, gold_path).history().select(
    "version", "timestamp", "operation", "operationParameters"
).show(truncate=False)

+-------+-----------------------+---------+--------------------------------------+
|version|timestamp              |operation|operationParameters                   |
+-------+-----------------------+---------+--------------------------------------+
|0      |2026-08-10 08:05:18.037|WRITE    |{mode -> Overwrite, partitionBy -> []}|
+-------+-----------------------+---------+--------------------------------------+

